# MCP Documentation Extraction with Griffe

This notebook uses Griffe for advanced Python API documentation extraction, combined with fenic's semantic operations for enhanced documentation generation.

In [1]:
# Install griffe if needed
# !pip install griffe

In [1]:
from typing import Literal, Optional, List, Dict, Any
from pydantic import BaseModel, Field
import fenic as fc
import griffe
import json
from pathlib import Path

In [2]:
# Configure fenic session
config = fc.SessionConfig(
        app_name="docs",
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
                "flash-lite": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash-lite",
                    rpm=4000,
                    tpm=4_000_000,
                ),
                "mini": fc.OpenAIModelConfig(
                    model_name="gpt-4.1-mini",
                    rpm=500,
                    tpm=200_000,
                ),
                "nano": fc.OpenAIModelConfig(
                    model_name="gpt-4.1-nano",
                    rpm=500,
                    tpm=200_000,
                ),
            },
            default_language_model="flash",
            embedding_models={
                "large": fc.OpenAIModelConfig(
                    model_name="text-embedding-3-large",
                    rpm=3000,
                    tpm=1_000_000
                )
            }
        ),
    )

session = fc.Session.get_or_create(config)

In [4]:
# Load fenic API using Griffe
loader = griffe.GriffeLoader()
fenic_api = loader.load("fenic")

print(f"Loaded module: {fenic_api.name}")
print(f"Module path: {fenic_api.filepath}")

Loaded module: fenic
Module path: /Users/kostaspardalis/Projects/fenic/src/fenic/__init__.py


In [ ]:
from typing import List, Dict, Any
import griffe
from griffe import Kind  # Griffe 1.x

def extract_api_elements(module: griffe.Module, parent_path: str = "") -> List[Dict[str, Any]]:
    elements: List[Dict[str, Any]] = []
    current_path = f"{parent_path}.{module.name}" if parent_path else module.name

    elements.append({
        "type": "module",
        "name": module.name,
        "qualified_name": current_path,
        "docstring": module.docstring.value if module.docstring else None,
        "filepath": str(module.filepath) if module.filepath else None,
        "is_public": module.is_public,
        "is_private": module.is_private,
        "line_start": module.lineno,
        "line_end": module.endlineno,
    })

    for member in module.members.values():
        if isinstance(member, griffe.Module):
            elements.extend(extract_api_elements(member, current_path))
        elif isinstance(member, griffe.Class):
            elements.append({
                "type": "class",
                "name": member.name,
                "qualified_name": f"{current_path}.{member.name}",
                "docstring": member.docstring.value if member.docstring else None,
                "bases": [str(b) for b in member.bases],
                "is_public": member.is_public,
                "is_private": member.is_private,
                "line_start": member.lineno,
                "line_end": member.endlineno,
            })
            for func in member.members.values():
                if isinstance(func, griffe.Function):
                    elements.append({
                        "type": "method",
                        "name": func.name,
                        "qualified_name": f"{current_path}.{member.name}.{func.name}",
                        "parent_class": member.name,
                        "docstring": func.docstring.value if func.docstring else None,
                        "is_public": func.is_public,
                        "is_private": func.is_private,
                        "is_property": getattr(func, "is_property", False),
                        "is_staticmethod": getattr(func, "is_staticmethod", False),
                        "is_classmethod": getattr(func, "is_classmethod", False),
                        "is_async": getattr(func, "async_", False),
                        "parameters": [p.name for p in func.parameters],
                        "returns": str(func.returns) if func.returns else None,
                        "line_start": func.lineno,
                        "line_end": func.endlineno,
                    })
        elif isinstance(member, griffe.Function):
            elements.append({
                "type": "function",
                "name": member.name,
                "qualified_name": f"{current_path}.{member.name}",
                "docstring": member.docstring.value if member.docstring else None,
                "is_public": member.is_public,
                "is_private": member.is_private,
                "is_async": getattr(member, "async_", False),
                "parameters": [p.name for p in member.parameters],
                "returns": str(member.returns) if member.returns else None,
                "line_start": member.lineno,
                "line_end": member.endlineno,
                # No is_property, is_staticmethod, is_classmethod here!
            })
        elif isinstance(member, griffe.Attribute):
            elements.append({
                "type": "attribute",
                "name": member.name,
                "qualified_name": f"{current_path}.{member.name}",
                "docstring": member.docstring.value if member.docstring else None,
                "value": str(member.value) if member.value else None,
                "annotation": str(member.annotation) if member.annotation else None,
                "is_public": member.is_public,
                "is_private": member.is_private,
                "line_start": member.lineno,
                "line_end": member.endlineno,
            })

    return elements

In [ ]:
# Extract all API elements
api_elements = extract_api_elements(fenic_api)
print(f"Extracted {len(api_elements)} API elements")

# Create DataFrame
api_df = session.createDataFrame(api_elements)

# Show breakdown by type
print("\nAPI Elements by Type:")
api_df.group_by("type").agg(
    fc.count("*").alias("count")
).order_by(fc.col("count").desc()).show()

In [7]:
# 1. Extract hierarchy levels from qualified_name
hierarchy_df = api_df.select(
      "*",
      # Split the qualified name into parts
      fc.text.split(fc.col("qualified_name"), r"\.").alias("path_parts"),

      # Get the depth (number of dots + 1)
      (fc.text.length(fc.col("qualified_name")) -
      fc.text.length(fc.text.regexp_replace(fc.col("qualified_name"), r"\.", "")) + 1).alias("depth")
  )

# 2. Create parent qualified names
parent_df = hierarchy_df.select(
      "*",
      # For each element, derive its parent's qualified_name
      fc.when(
          fc.col("depth") > 1,
          fc.text.regexp_replace(fc.col("qualified_name"), r"\.[^.]+$", "")
      ).otherwise(fc.lit("None")).alias("parent_qualified_name")
  )

# 3. Create a module column from qualified_name
'''with_module_df = parent_df.select(
      "*",
      # Extract module based on type
      fc.when(
          fc.col("type") == "module",
          fc.col("qualified_name")
      ).when(
          fc.col("type").is_in(["method", "async_method"]),
          # For methods, module is everything before the class name
          fc.text.regexp_replace(fc.col("qualified_name"), r"\.[^.]+\.[^.]+$", "")
      ).when(
          fc.col("type").is_in(["class", "function"]),
          # For classes/functions, module is everything before the last part
          fc.text.regexp_replace(fc.col("qualified_name"), r"\.[^.]+$", "")
      ).otherwise(fc.lit("None")).alias("module")
  )'''

with_module_df = parent_df.select(
      "*",
      # Extract module based on type and qualified_name
      fc.when(
          fc.col("type") == "module",
          fc.col("qualified_name")
      ).when(
          (fc.col("type").is_in(["method", "async_method"])) & (fc.col("depth") > 2),
          # For methods: module.submodule.Class.method -> module.submodule
          fc.text.regexp_replace(fc.col("qualified_name"), r"\.[^.]+\.[^.]+$", "")
      ).when(
          (fc.col("type").is_in(["class", "function", "attribute"])) & (fc.col("depth") > 1),
          # For classes/functions: module.submodule.Class -> module.submodule
          fc.text.regexp_replace(fc.col("qualified_name"), r"\.[^.]+$", "")
      ).when(
          fc.col("depth") == 1,
          fc.col("qualified_name")  # Top-level items are their own module
      ).otherwise(
          # Extract everything before the last component for depth 2+
          fc.text.regexp_replace(fc.col("qualified_name"), r"\.[^.]+$", "")
      ).alias("module")
  )

  # Create renamed dataframes with all columns aliased
child_df_renamed = with_module_df.select(
      *[fc.col(c).alias(f"{c}_child") for c in with_module_df.columns if c != "parent_qualified_name"],
      fc.col("parent_qualified_name").alias("join_key")
  )

parent_df_renamed = with_module_df.select(
      *[fc.col(c).alias(f"{c}_parent") for c in with_module_df.columns if c != "qualified_name"],
      fc.col("qualified_name").alias("join_key")
  )

  # Now join on the common column name
hierarchy_relationships = child_df_renamed.join(
      parent_df_renamed,
      on="join_key",
      how="left"
).select(
      fc.col("qualified_name_child").alias("child_name"),
      fc.col("type_child").alias("child_type"),
      fc.col("name_child").alias("child_short_name"),
      fc.col("join_key").alias("parent_name"),
      fc.col("type_parent").alias("parent_type"),
      fc.col("name_parent").alias("parent_short_name")
)

# 5. Create tree structure by module
module_tree = with_module_df.group_by("module").agg(
      fc.count("*").alias("element_count"),
      fc.collect_list(
          fc.struct(
              fc.col("qualified_name"),
              fc.col("type"),
              fc.col("name"),
              fc.col("parent_class")
          )
      ).alias("elements")
  )

# 6. Get class hierarchy with methods
# First, prepare the dataframes with matching column names
classes_df = with_module_df.filter(
      fc.col("type") == "class"
  ).select(
      fc.col("qualified_name").alias("class_qualified_name"),
      fc.col("name").alias("class_name"),
      fc.col("docstring").alias("class_docstring"),
      fc.col("name").alias("join_key")  # This will match with parent_class
  )

methods_df = with_module_df.filter(
      fc.col("type").is_in(["method", "async_method"])
  ).select(
      fc.col("name").alias("method_name"),
      fc.col("parent_class").alias("join_key")  # This will match with class name
  )
  
# Now join on the common column
class_hierarchy = classes_df.join(
      methods_df,
      on="join_key",
      how="left"
  ).group_by(
      "class_qualified_name",
      "class_name",
      "class_docstring"
  ).agg(
      fc.count("method_name").alias("method_count"),
      fc.collect_list("method_name").alias("methods")
  )
  
# 7. Create a navigable structure
navigation_df = with_module_df.select(
      "qualified_name",
      "type",
      "name",
      "parent_qualified_name",
      "module",
      "depth",
      # Create a sort key for hierarchical ordering
      fc.text.concat_ws(
          "_",
          fc.col("module"),
          fc.when(fc.col("type") == "module", fc.lit("0"))
            .when(fc.col("type") == "class", fc.lit("1"))
            .when(fc.col("type").is_in(["function", "attribute"]), fc.lit("2"))
            .when(fc.col("type").is_in(["method", "async_method"]), fc.lit("3"))
            .otherwise(fc.lit("4")),
          fc.col("name")
      ).alias("sort_key")
  ).order_by("sort_key")

In [8]:
# 1. Filter to leaves (methods and functions)
leaves_df = api_df.filter(
      fc.col("type").is_in(["method", "function", "async_method"])
  )

# 2. Create enriched context for each method/function
  # Use array.join to convert parameters array to string
leaves_with_params_str = leaves_df.select(
      "*",
      fc.when(
          fc.col("parameters").is_not_null() & (fc.array_size(fc.col("parameters")) > 0),
          fc.text.array_join(fc.col("parameters"), ", ")
      ).otherwise(fc.lit("")).alias("parameters_str")
  )

  # Now create the full context
leaves_with_context = leaves_with_params_str.select(
      "*",
      fc.text.concat_ws(
          "\n",
          fc.lit("Name:"), fc.col("name"),
          fc.lit("Type:"), fc.col("type"),
          fc.lit("Parent Class:"), fc.coalesce(fc.col("parent_class"), fc.lit("None")),
          fc.lit("Qualified Name:"), fc.col("qualified_name"),
          fc.lit("Parameters:"), fc.col("parameters_str"),
          fc.lit("Returns:"), fc.coalesce(fc.col("returns"), fc.lit("None")),
          fc.lit("Docstring:"), fc.coalesce(fc.col("docstring"), fc.lit("No documentation"))
      ).alias("full_context")
  )

In [9]:
# 3. Define structured summary schema
class MethodSummary(BaseModel):
      """Structured summary of a method or function."""
      purpose: str = Field(description="What this method/function does in one clear sentence")
      category: Literal[
          "data_transformation",
          "io_operation",
          "configuration",
          "utility",
          "lifecycle",
          "query",
          "semantic_operation",
          "internal",
          "aggregation",
          "schema_operation"
      ] = Field(description="The primary category of this method")
      inputs: str = Field(description="Simple description of inputs (e.g., 'column name and value')")
      outputs: str = Field(description="Simple description of output (e.g., 'filtered DataFrame')")
      usage_pattern: str = Field(description="When/how to use this (e.g., 'Use when filtering rows by condition')")  # 4. Generate summaries for methods (let's start with a sample)
  # Focus on public methods first
public_methods = leaves_with_context.filter(
      fc.col("is_public") == True
  )

method_summaries = public_methods.select(
      "qualified_name",
      "name",
      "parent_class",
      "full_context",
      fc.semantic.extract(
          fc.col("full_context"),
          MethodSummary,
          model_alias="flash"
      ).alias("summary")
  ).cache()



In [ ]:
categorized_methods =method_summaries.unnest("summary").group_by(
      fc.col("category")
  ).agg(
      fc.count("*").alias("count"),
      fc.collect_list(fc.struct(
          fc.col("name"),
          fc.col("parent_class"),
          fc.col("purpose")
      )).alias("methods")
  )
  
# 6. Analyze methods by parent class to understand class behavior
class_method_analysis = method_summaries.unnest("summary").filter(
      fc.col("parent_class").is_not_null()
  ).group_by("parent_class").agg(
      fc.count("*").alias("method_count"),
      fc.collect_list(fc.col("category")).alias("method_categories"),
      fc.collect_list(fc.struct(
          fc.col("name"),
          fc.col("purpose")
      )).alias("method_purposes")
  )

print("\nClass behavior analysis:")
class_method_analysis.select(
      "parent_class",
      "method_count",
      "method_categories"
  ).show(5)

In [11]:
# 7. Identify semantic operations specifically
semantic_methods = leaves_with_context.filter(
      fc.col("qualified_name").contains("semantic")
  )

# Generate specialized summaries for semantic operations
class SemanticOperationSummary(BaseModel):
      """Summary specifically for semantic/LLM operations."""
      operation_type: Literal["map", "extract", "group_by", "cluster", "classify", "join", "filter", "aggregate", "embed"] = Field(description="The type of semantic operation this is")
      llm_usage: str = Field(description="How this operation uses LLMs")
      typical_use_case: str = Field(description="Real-world example of when to use this")
      input_requirements: str = Field(description="What kind of input data this expects")
      output_format: str = Field(description="What the LLM operation produces")

semantic_summaries = semantic_methods.select(
      "qualified_name",
      "name",
      "full_context",
      fc.semantic.extract(
          fc.col("full_context"),
          SemanticOperationSummary,
          model_alias="flash"
      ).alias("semantic_summary")
  ).cache()

In [ ]:
# 8. Look for method naming patterns
# Use regexp_replace to identify patterns
method_prefixes = leaves_df.select(
      fc.col("name"),
      # Extract prefix by removing everything after first underscore
      fc.when(
          fc.col("name").contains("_"),
          fc.text.split(fc.col("name"), "_").get_item(0)
      ).otherwise(fc.lit("")).alias("prefix")
  ).filter(
      fc.col("prefix") != ""
  ).group_by("prefix").agg(fc.count("*").alias("count")).order_by(fc.col("count").desc())

print("\nCommon method prefixes:")
method_prefixes.show(10)

In [13]:
  # 9. Create a "method signature" summary for complex methods
complex_methods = leaves_with_context.filter(
      fc.array_size(fc.col("parameters")) > 3  # Methods with many parameters
)

# Define schema using ExtractSchema with list support
method_signature_schema = fc.ExtractSchema([
      fc.ExtractSchemaField(
          name="simplified_signature",
          data_type=fc.StringType,
          description="Simplified method signature with key parameters"
      ),
      fc.ExtractSchemaField(
          name="required_params",
          data_type=fc.ExtractSchemaList(element_type=fc.StringType),
          description="List of required parameters"
      ),
      fc.ExtractSchemaField(
          name="optional_params",
          data_type=fc.ExtractSchemaList(element_type=fc.StringType),
          description="List of optional parameters with defaults"
      ),
      fc.ExtractSchemaField(
          name="complexity_note",
          data_type=fc.StringType,
          description="Note about method complexity or special usage"
      )
  ])

# Generate signatures for complex methods
complex_method_signatures = complex_methods.select(
      "qualified_name",
      "name",
      "parameters",
      "full_context",
      fc.semantic.extract(
          fc.col("full_context"),
          method_signature_schema,
          model_alias="flash"
      ).alias("signature_analysis")
  ).cache()


In [1]:
print("\nComplex method signature analysis:")
complex_method_signatures.select(
      "name",
      fc.array_size(fc.col("parameters")).alias("param_count"),
      "signature_analysis"
  ).show(5)


Complex method signature analysis:


NameError: name 'complex_method_signatures' is not defined

In [ ]:
complex_method_signatures.write.save_as_table("complex_method_signatures", mode="overwrite")
method_prefixes.write.save_as_table("method_prefixes", mode="overwrite")
semantic_summaries.write.save_as_table("semantic_summaries", mode="overwrite")
class_method_analysis.write.save_as_table("class_method_analysis", mode="overwrite")
categorized_methods.write.save_as_table("categorized_methods", mode="overwrite")
method_summaries.write.save_as_table("method_summaries", mode="overwrite")
leaves_with_context.write.save_as_table("leaves_with_context", mode="overwrite")
navigation_df.write.save_as_table("navigation_df", mode="overwrite")
class_hierarchy.write.save_as_table("class_hierarchy", mode="overwrite")
module_tree.write.save_as_table("module_tree", mode="overwrite")
hierarchy_relationships.write.save_as_table("hierarchy_relationships", mode="overwrite")
with_module_df.write.save_as_table("with_module_df", mode="overwrite")
parent_df.write.save_as_table("parent_df", mode="overwrite")
hierarchy_df.write.save_as_table("hierarchy_df", mode="overwrite")
api_df.write.save_as_table("api_df", mode="overwrite")

In [36]:
# Get all classes with their basic info
classes_df = api_df.filter(
    fc.col("type") == "class"
)

  # Prepare classes with renamed columns
classes_for_join = classes_df.select(
      fc.col("qualified_name").alias("class_qualified_name"),
      fc.col("name").alias("class_name"),
      fc.col("docstring").alias("class_docstring"),
      fc.col("bases"),
      fc.col("is_public"),
      fc.col("is_private"),
      fc.col("name").alias("class_join_key")  # Keep original name for join
  )

  # Prepare methods with only the columns we need
methods_for_join = method_summaries.filter(
      fc.col("parent_class").is_not_null()
  ).select(
      fc.col("name").alias("method_name"),
      fc.col("summary"),
      fc.col("parent_class").alias("class_join_key")
  )

  # Join with the method summaries
classes_with_methods = classes_for_join.join(
      methods_for_join,
      on="class_join_key",
      how="left"
  )

# Aggregate method information per class
class_method_aggregates = classes_with_methods.unnest("summary").group_by(
      "class_qualified_name",
      "class_name",
      "class_docstring",
      "bases",
      "is_public",
      "is_private"
  ).agg(
      fc.count(fc.col("method_name")).alias("method_count"),
      fc.collect_list(fc.struct(
          fc.col("purpose"),
          fc.col("category"),
          fc.col("usage_pattern")
      )).alias("method_summaries"),
      # Get distribution of method categories
      fc.collect_list(fc.col("category")).alias("method_categories")
  )
  
class ClassSummary(BaseModel):
      """High-level summary of a class's purpose and role."""
      primary_purpose: str = Field(description="Main purpose of this class in one sentence")
      role: Literal[
          "data_container",     # Holds data (like DataFrame)
          "operator",          # Performs operations
          "configuration",     # Config/settings classes
          "factory",          # Creates other objects
          "manager",          # Manages resources/lifecycle
          "utility",          # Helper/utility class
          "exception",        # Error handling
          "interface",        # Abstract base class
          "model"            # Data model/schema
      ] = Field(description="The architectural role of this class")
      key_capabilities: str = Field(description="2-3 main capabilities based on its methods")
      usage_context: str = Field(description="When and how to use this class")
      relationships: str = Field(description="Key relationships with other classes")

class_summaries = class_method_aggregates.select(
      "*",
      fc.text.array_join(fc.col("bases"), ", ").alias("bases_str"),
      fc.text.array_join(fc.col("method_categories"), ", ").alias("method_categories_str")
  ).select(
      "*",
      fc.text.concat_ws(
          "\n",
          fc.lit("Class:"), fc.col("class_name"),
          fc.lit("Qualified Name:"), fc.col("class_qualified_name"),
          fc.lit("Inherits from:"), fc.col("bases_str"),
          fc.lit("Docstring:"), fc.coalesce(fc.col("class_docstring"), fc.lit("No documentation")),
          fc.lit("Method count:"), fc.col("method_count").cast(fc.StringType),
          fc.lit("Method categories:"), fc.col("method_categories_str")
      ).alias("class_context"),
      fc.semantic.extract(
          fc.text.concat_ws(
              "\n",
              fc.lit("Class:"), fc.col("class_name"),
              fc.lit("Docstring:"), fc.coalesce(fc.col("class_docstring"), fc.lit("No documentation")),
              fc.lit("Method count:"), fc.col("method_count").cast(fc.StringType),
              fc.lit("Method categories:"), fc.col("method_categories_str")
          ),
          ClassSummary,
          model_alias="flash"
      ).alias("class_summary")
  ).cache()

In [ ]:
summed_classes = class_summaries.select(fc.col("class_name").alias("name"), fc.col("class_summary")).unnest("class_summary")
summed_classes_with_module = with_module_df.select("name", "module", "type").join(summed_classes, on="name", how="right")

summed_classes_with_module.show()

In [ ]:
method_summaries_with_modules = with_module_df.select("name", "module", "type").join(method_summaries, on="name", how="right").select("module", "type", "name", "summary").unnest("summary")

In [76]:
module_method_summaries = method_summaries_with_modules.group_by("module").agg(fc.semantic.reduce("summarize the purpose of the module, based on the methods in the module. Here's the metadata for the method. name: {name}, purpose: {purpose}, usage_pattern: {usage_pattern} ").alias("module_method_summary")).cache()

In [77]:
module_class_summaries = summed_classes_with_module.group_by("module").agg(fc.semantic.reduce("ummarize the purpose of the module, based on the classes in the module. Here's the metadata for the class. name: {name}, purpose: {primary_purpose}, key capabilities: {key_capabilities}, usage context: {usage_context}, relationships: {relationships} ").alias("module_class_summary")).cache()


In [ ]:
module_method_summaries.show()

In [ ]:
module_class_summaries.show()

In [94]:
modules = module_class_summaries.join(module_method_summaries, on="module", how="right")
modules_without_nulls = modules.with_column("module_class_summary", fc.coalesce(fc.col("module_class_summary"), fc.lit("None")))
modules_with_summaries = modules_without_nulls.with_column("summary", fc.semantic.map("summarize the purpose of the module based on the summary of their classes {module_class_summary} and methods {module_method_summary}")).cache()

In [ ]:
modules_with_summaries.show()

In [98]:
fenic_summary = modules_with_summaries.agg(fc.semantic.reduce("summarize the purpose and capabilities of the project based on the summaries of the modules {summary}").alias("project_summary")).cache()

In [ ]:
fenic_summary.write.save_as_table("fenic_summary")
module_method_summaries.write.save_as_table("module_method_summaries")
module_class_summaries.write.save_as_table("module_class_summaries")
modules_with_summaries.write.save_as_table("modules_with_summaries")

In [110]:
api_df.filter((fc.col("is_public") == True) & (fc.col("type") != "attribute")).select("type", "name", "qualified_name", "docstring").show()

┌──────────┬──────────────────────┬────────────────────────────────┬───────────────────────────────┐
│ type     ┆ name                 ┆ qualified_name                 ┆ docstring                     │
╞══════════╪══════════════════════╪════════════════════════════════╪═══════════════════════════════╡
│ module   ┆ fenic                ┆ fenic                          ┆ Fenic is an opinionated,      │
│          ┆                      ┆                                ┆ PySpark-inspired DataFrame    │
│          ┆                      ┆                                ┆ framework for building        │
│          ┆                      ┆                                ┆ production AI and agentic     │
│          ┆                      ┆                                ┆ applications.                 │
│ function ┆ py_validate_jq_query ┆ fenic._polars_plugins.py_valid ┆ null                          │
│          ┆                      ┆ ate_jq_query                   ┆                       

In [20]:
hierarchy = session.table("hierarchy_df").filter((fc.col("is_public") == True) & (fc.col("type") != "attribute") & (~fc.col("name").starts_with("_")))
py_hieararchy = hierarchy.select("qualified_name", "name", "type", "depth", "path_parts").to_pydict()

def build_tree(hierarchy_dict):
      """Build a tree structure from the flat hierarchy data."""
      tree = {"name": "fenic", "type": "root", "children": {}}

      # Process each element
      for i, qual_name in enumerate(hierarchy_dict['qualified_name']):
          name = hierarchy_dict['name'][i]
          elem_type = hierarchy_dict['type'][i]
          depth = hierarchy_dict['depth'][i]
          path_parts = hierarchy_dict['path_parts'][i]

          # Navigate to the correct position in the tree
          current = tree
          for j, part in enumerate(path_parts[:-1]):  # All but the last part
              if part not in current['children']:
                  current['children'][part] = {
                      "name": part,
                      "type": "unknown",  # Will be updated when we process that element
                      "children": {}
                  }
              current = current['children'][part]

          # Add the final element
          if len(path_parts) > 0:
              final_part = path_parts[-1]
              current['children'][final_part] = {
                  "name": name,
                  "type": elem_type,
                  "qualified_name": qual_name,
                  "depth": depth,
                  "children": {}
              }

      return tree

api_tree = build_tree(py_hieararchy)

  # Function to print tree nicely
def print_tree(node, indent=0, max_depth=3):
      """Print tree structure with indentation."""
      if indent > max_depth:
          return

      if indent > 0:  # Skip root
          print("  " * (indent-1) + f"├─ [{node['type']}] {node['name']}")

      # Sort children by type then name for better readability
      children = sorted(
          node.get('children', {}).values(),
          key=lambda x: (
              0 if x['type'] == 'module' else
              1 if x['type'] == 'class' else
              2 if x['type'] == 'function' else
              3 if x['type'] == 'method' else
              4,
              x['name']
          )
      )

      for child in children[:10]:  # Limit to first 10 children
          print_tree(child, indent + 1, max_depth)

      if len(children) > 10:
          print("  " * indent + f"... and {len(children) - 10} more")

print_tree(api_tree)

├─ [module] fenic
  ├─ [module] api
    ├─ [module] catalog
    ├─ [module] column
    ├─ [module] dataframe
    ├─ [module] functions
    ├─ [module] io
    ├─ [module] lineage
    ├─ [module] session
    ├─ [module] window
  ├─ [module] core
    ├─ [module] error
      ... and 10 more
    ├─ [module] metrics
    ├─ [module] types
    ├─ [unknown] _interfaces
    ├─ [unknown] _logical_plan
    ├─ [unknown] _resolved_session_config
    ├─ [unknown] _utils
  ├─ [module] logging
    ├─ [function] configure_logging
  ├─ [unknown] _inference
    ├─ [module] anthropic
    ├─ [module] embedding_model
    ├─ [module] google
    ├─ [module] language_model
    ├─ [module] model_catalog
    ├─ [module] model_client
      ... and 2 more
    ├─ [module] openai
    ├─ [module] token_counter
    ├─ [module] types
  ├─ [unknown] _polars_plugins
    ├─ [function] py_validate_jq_query


In [10]:
modules = session.table("modules_with_summaries").select("module", "summary")

modules.filter(fc.col("summary").contains_any(["join"])).show()

┌─────────────────────────────────────┬────────────────────────────────────────────────────────────┐
│ module                              ┆ summary                                                    │
╞═════════════════════════════════════╪════════════════════════════════════════════════════════════╡
│ fenic.core._logical_plan.plans.join ┆ The module's purpose is to provide classes                 │
│                                     ┆ [MODULE_CLASS_SUMMARY] for joining data from multiple      │
│                                     ┆ sources based on specified conditions or semantic          │
│                                     ┆ relationships. This involves initializing objects,         │
│                                     ┆ configuring data processing, managing performance metrics, │
│                                     ┆ handling exceptions, configuring language models and cloud │
│                                     ┆ settings, managing query results and schemas, optim

In [20]:
funcs = session.table("method_summaries").filter(~fc.col("name").starts_with("_")).select("name","qualified_name", "summary").unnest("summary").select("name", "qualified_name", "purpose", "usage_pattern")

In [25]:
funcs.filter(~fc.col("qualified_name").rlike(r"\._")).count()

177

In [19]:
raw = session.table("api_df")

raw.filter(fc.col("name") =="_polars_plugins" ).select("name", "qualified_name", "is_public").show()

┌─────────────────┬───────────────────────┬───────────┐
│ name            ┆ qualified_name        ┆ is_public │
╞═════════════════╪═══════════════════════╪═══════════╡
│ _polars_plugins ┆ fenic._polars_plugins ┆ false     │
└─────────────────┴───────────────────────┴───────────┘


In [16]:
def search_code(
      query: str,
      search_mode: str = "all",
      element_type: Optional[str] = None,
      include_private: bool = False,
      max_results: int = 50,
      case_sensitive: bool = False
  ) -> str:
      """
      Search the Fenic codebase using pre-extracted metadata.

      Args:
          query: The search term or regex pattern to look for
          search_mode: Where to search - "name", "docs", "code", or "all" (default: "all")
          element_type: Filter by type - "function", "class", "method", "attribute", or None for all
          include_private: Whether to include private elements (starting with _) (default: False)
          max_results: Maximum number of results to return (default: 50)
          case_sensitive: Whether the search should be case sensitive (default: False)

      Returns:
          Formatted search results with element details and context

      Examples:
          - Search for functions: query="extract", element_type="function"
          - Search in docstrings: query="embedding", search_mode="docs"
          - Search for classes: query="DataFrame", element_type="class"
          - Regex search: query="semantic\\..*", search_mode="name"
      """
      try:
          # Get session and base table
          df = session.table("api_df")

          # Apply privacy filter
          if not include_private:
              df = df.filter(fc.col("is_public") == True)
              # Also filter out private modules
              df = df.filter(~fc.col("qualified_name").contains("._"))

          # Apply type filter
          if element_type:
              df = df.filter(fc.col("type") == element_type)

          # Prepare regex pattern
          if not case_sensitive:
              pattern = f"(?i){query}"
          else:
              pattern = query

          # Apply search based on mode
          if search_mode == "name":
              # Search in name and qualified_name
              search_df = df.filter(
                  fc.col("name").rlike(pattern) |
                  fc.col("qualified_name").rlike(pattern)
              )

          elif search_mode == "docs":
              # Search in docstrings
              search_df = df.filter(
                  fc.col("docstring").is_not_null() &
                  fc.col("docstring").rlike(pattern)
              )

          elif search_mode == "code":
              # Search in value, annotation, returns
              search_df = df.filter(
                  (fc.col("value").is_not_null() & fc.col("value").rlike(pattern)) |
                  (fc.col("annotation").is_not_null() & fc.col("annotation").rlike(pattern)) |
                  (fc.col("returns").is_not_null() & fc.col("returns").rlike(pattern))
              )

          else:  # "all" mode
              # Search across all relevant text columns
              search_df = df.filter(
                  fc.col("name").rlike(pattern) |
                  fc.col("qualified_name").rlike(pattern) |
                  (fc.col("docstring").is_not_null() & fc.col("docstring").rlike(pattern)) |
                  (fc.col("value").is_not_null() & fc.col("value").rlike(pattern)) |
                  (fc.col("annotation").is_not_null() & fc.col("annotation").rlike(pattern)) |
                  (fc.col("returns").is_not_null() & fc.col("returns").rlike(pattern))
              )

            # Add relevance scoring
          search_df = search_df.select(
                "*",
                fc.when(fc.col("name").rlike(pattern), fc.lit(10)).otherwise(fc.lit(0)).alias("name_score"),
                fc.when(fc.col("qualified_name").rlike(pattern), fc.lit(5)).otherwise(fc.lit(0)).alias("qname_score"),
                fc.when(fc.col("docstring").is_not_null() & fc.col("docstring").rlike(pattern),
            fc.lit(3)).otherwise(fc.lit(0)).alias("doc_score"),
                fc.when(fc.col("value").is_not_null() & fc.col("value").rlike(pattern), fc.lit(2)).otherwise(fc.lit(0)).alias("value_score"),
                fc.when(fc.col("annotation").is_not_null() & fc.col("annotation").rlike(pattern),
            fc.lit(2)).otherwise(fc.lit(0)).alias("annotation_score"),
                fc.when(fc.col("returns").is_not_null() & fc.col("returns").rlike(pattern),
            fc.lit(2)).otherwise(fc.lit(0)).alias("returns_score")
            )

          # Calculate total relevance score
          search_df = search_df.select(
              "*",
              (fc.col("name_score") + fc.col("qname_score") +
               fc.col("doc_score") + fc.col("value_score") +
               fc.col("annotation_score") + fc.col("returns_score")).alias("relevance_score")
          )

            # Sort by relevance and type, then limit
          search_df = search_df.order_by(
              [fc.col("relevance_score").desc(),
              fc.col("type"),
              fc.col("qualified_name")]
          ).limit(max_results)

          # Collect results
          results = search_df.to_pydict()

          # Format output
          output = f"# Search Results for '{query}'\n\n"
          output += f"**Search Mode**: {search_mode} | "
          output += f"**Element Type**: {element_type or 'all'} | "
          output += f"**Results Found**: {len(results.get('name', []))}\n\n"

          if not results.get('name'):
              output += "No matches found.\n"
              return output

          # Group results by type for better organization
          current_type = None

          for i in range(len(results['name'])):
              # Add type header if it changed
              if results['type'][i] != current_type:
                  current_type = results['type'][i]
                  output += f"\n## {current_type.capitalize()}s\n\n"

              # Element header
              output += f"### `{results['name'][i]}`\n"

              # Basic info
              output += f"**Full Path**: `{results['qualified_name'][i]}`\n"

              if results.get('filepath') and results['filepath'][i]:
                  line_info = f"{results['filepath'][i]}"
                  if results.get('line_start') and results['line_start'][i]:
                      line_info += f":{results['line_start'][i]}"
                      if results.get('line_end') and results['line_end'][i]:
                          line_info += f"-{results['line_end'][i]}"
                  output += f"**Location**: {line_info}\n"

              # Type-specific information
              if results['type'][i] in ['function', 'method']:
                  # Parameters
                  if results.get('parameters') and results['parameters'][i]:
                      params = results['parameters'][i]
                      if isinstance(params, list) and params:
                          output += f"**Parameters**: `{', '.join(params)}`\n"

                  # Return type
                  if results.get('returns') and results['returns'][i]:
                      output += f"**Returns**: `{results['returns'][i]}`\n"

                  # Async indicator
                  if results.get('is_async') and results['is_async'][i]:
                      output += f"**Async**: Yes\n"

              elif results['type'][i] == 'class':
                  # Base classes
                  if results.get('bases') and results['bases'][i]:
                      bases = results['bases'][i]
                      if isinstance(bases, list) and bases:
                          output += f"**Inherits from**: `{', '.join(bases)}`\n"

              elif results['type'][i] == 'method':
                  # Method type
                  if results.get('is_property') and results['is_property'][i]:
                      output += f"**Method Type**: Property\n"
                  elif results.get('is_staticmethod') and results['is_staticmethod'][i]:
                      output += f"**Method Type**: Static Method\n"
                  elif results.get('is_classmethod') and results['is_classmethod'][i]:
                      output += f"**Method Type**: Class Method\n"

              # Docstring (truncated if too long)
              if results.get('docstring') and results['docstring'][i]:
                  doc = results['docstring'][i].strip()
                  if len(doc) > 200:
                      doc = doc[:197] + "..."
                  # Extract first line for summary
                  first_line = doc.split('\n')[0]
                  output += f"**Description**: {first_line}\n"

              # Relevance score (for debugging)
              if results.get('relevance_score') and results['relevance_score'][i] > 0:
                  output += f"**Relevance**: {results['relevance_score'][i]}\n"

              output += "\n"

          # Add search tips if no or few results
          if len(results.get('name', [])) < 5:
              output += "\n---\n"
              output += "**Search Tips**:\n"
              output += "- Try using regex patterns (e.g., `semantic.*join` for semantic join functions)\n"
              output += "- Use search_mode='docs' to search only in documentation\n"
              output += "- Set include_private=True to include private methods\n"
              output += "- Try broader search terms or different variations\n"

          return output

      except Exception as e:
          return f"Error during search: {str(e)}\n\nPlease check that the query is valid and try again."

In [18]:
print(search_code("join"))

# Search Results for 'join'

**Search Mode**: all | **Element Type**: all | **Results Found**: 12


## Attributes

### `JoinType`
**Full Path**: `fenic.core.types.enums.JoinType`
**Description**: Type alias representing supported join types.
**Relevance**: 18


## Classs

### `JoinExample`
**Full Path**: `fenic.core.types.semantic_examples.JoinExample`
**Inherits from**: `BaseModel`
**Description**: A single semantic example for semantic join operations.
**Relevance**: 18

### `JoinExampleCollection`
**Full Path**: `fenic.core.types.semantic_examples.JoinExampleCollection`
**Inherits from**: `BaseExampleCollection[JoinExample]`
**Description**: Collection of examples for semantic join operations.
**Relevance**: 18


## Functions

### `array_join`
**Full Path**: `fenic.api.functions.text.array_join`
**Parameters**: `column, delimiter`
**Returns**: `Column`
**Description**: Joins an array of strings into a single string with a delimiter.
**Relevance**: 18


## Methods

### `join`
**Full 

In [24]:
def search(query: str, max_results: int = 30) -> str:
      """
      Search the Fenic codebase for functions, classes, methods, and other code elements.

      Args:
          query: Search term or regex pattern to find in code names, documentation, and signatures
          max_results: Maximum number of results to return (default: 30)

      Returns:
          Search results with type, name, qualified path, and brief description

      Examples:
          - Simple search: "join"
          - Regex search: "semantic.*extract"
          - Search for specific terms: "DataFrame"
      """
      try:
          df = session.table("api_df")

          # Filter only public API elements
          df = df.filter(
              (fc.col("is_public") == True) &
              (~fc.col("qualified_name").contains("._"))
          )

          # Search across all text fields
          search_df = df.filter(
              fc.col("name").rlike(f"(?i){query}") |
              fc.col("qualified_name").rlike(f"(?i){query}") |
              (fc.col("docstring").is_not_null() & fc.col("docstring").rlike(f"(?i){query}")) |
              (fc.col("annotation").is_not_null() & fc.col("annotation").rlike(f"(?i){query}")) |
              (fc.col("returns").is_not_null() & fc.col("returns").rlike(f"(?i){query}"))
          )

          # Add relevance scoring
          search_df = search_df.select(
              "type", "name", "qualified_name", "docstring",
              fc.when(fc.col("name").rlike(f"(?i){query}"), fc.lit(10)).otherwise(fc.lit(0)).alias("name_score"),
              fc.when(fc.col("qualified_name").rlike(f"(?i){query}"), fc.lit(5)).otherwise(fc.lit(0)).alias("path_score")
          )

          # Calculate total score and sort
          search_df = search_df.select(
              "*",
              (fc.col("name_score") + fc.col("path_score")).alias("score")
          ).order_by([fc.col("score").desc(), fc.col("type"), fc.col("name")]).limit(max_results)

          # Collect results
          results = search_df.to_pydict()

          # Format output
          output = f"# Search Results for: `{query}`\n\n"
          output += f"Found {len(results.get('name', []))} matches\n\n"

          if not results.get('name'):
              output += "No results found. Try:\n"
              output += "- Different keywords (e.g., 'extract', 'semantic', 'DataFrame')\n"
              output += "- Regex patterns (e.g., 'join.*semantic')\n"
              return output

          # Group by type for clarity
          current_type = None
          for i in range(len(results['name'])):
              if results['type'][i] != current_type:
                  current_type = results['type'][i]
                  output += f"\n## {current_type.capitalize()}s\n"

              # Format each result concisely
              output += f"\n**`{results['name'][i]}`** - `{results['qualified_name'][i]}`\n"

              # Add first line of docstring if available
              #if results.get('docstring') and results['docstring'][i]:
               #   first_line = results['docstring'][i].strip().split('\n')[0]
                #  if len(first_line) > 100:
                 #     first_line = first_line[:97] + "..."
                 # output += f"  {first_line}\n"
              if results.get('docstring') and results['docstring'][i]:
                  output += f"  {results['docstring'][i]}\n"
          #output += f"\n---\nUse `get_details(qualified_name)` to see full documentation for any result."

          return output

      except Exception as e:
          return f"Search error: {str(e)}"

In [25]:
print(search("join"))

# Search Results for: `join`

Found 12 matches


## Attributes

**`JoinType`** - `fenic.core.types.enums.JoinType`
  Type alias representing supported join types.

Valid values:

- "inner": Inner join, returns only rows that have matching values in both tables.
- "outer": Outer join, returns all rows from both tables, filling missing values with nulls.
- "left": Left join, returns all rows from the left table and matching rows from the right table.
- "right": Right join, returns all rows from the right table and matching rows from the left table.
- "cross": Cross join, returns the Cartesian product of the two tables.

## Classs

**`JoinExample`** - `fenic.core.types.semantic_examples.JoinExample`
  A single semantic example for semantic join operations.

Join examples demonstrate the evaluation of two input strings across different
datasets against a specific condition, used in a semantic.join operation.

**`JoinExampleCollection`** - `fenic.core.types.semantic_examples.JoinExampleColl

In [36]:
def initialize_learnings_table(session, include_embeddings: bool = True) -> bool:
    """
    Initialize the learnings table if it doesn't exist.
    
    Note: Fenic fully supports ArrayType in table schemas. The limitation about "primitive types only" 
    applies specifically to CSV import schemas, not table schemas in general.
    
    Args:
        session: Fenic session object
        include_embeddings: Whether to include embedding columns in the schema
        
    Returns:
        bool: True if table was created, False if it already existed
    """
    table_name = "learnings"
    
    # Check if table already exists
    if session.catalog.does_table_exist(table_name):
        return False
    
    # Define the base learnings table schema
    schema_fields = [
        fc.ColumnField('id', fc.StringType),
        fc.ColumnField('question', fc.StringType),
        fc.ColumnField('answer', fc.StringType),
        fc.ColumnField('learning_type', fc.StringType),
        fc.ColumnField('keywords', fc.ArrayType(fc.StringType)),  # Proper array type
        fc.ColumnField('related_functions', fc.ArrayType(fc.StringType)),  # Proper array type
        fc.ColumnField('created_at', fc.StringType)
    ]
    
    # Add embedding columns if requested
    if include_embeddings:
        # Note: You may need to adjust dimensions and model based on your embedding configuration
        # The dimensions shown here (3072) match the error message for text-embedding-3-large
        embedding_type = fc.EmbeddingType(dimensions=3072, embedding_model="openai/text-embedding-3-large")
        schema_fields.extend([
            fc.ColumnField('question_embedding', embedding_type),
            fc.ColumnField('answer_embedding', embedding_type),
            fc.ColumnField('combined_embedding', embedding_type)
        ])
    
    learnings_schema = fc.Schema(schema_fields)
    
    # Create the table
    session.catalog.create_table(table_name, learnings_schema)
    return True

In [37]:
import uuid
import datetime
def store_learning(
    session,
    question: str,
    answer: str,
    learning_type: str = "solution",  # "solution", "correction", "example"
    keywords: List[str] = None,
    related_functions: List[str] = None
) -> str:
    """
    Store a learning from a user interaction for future reference.
    
    Args:
        session: Fenic session object
        question: The original question or problem
        answer: The correct answer or solution
        learning_type: Type of learning (solution/correction/example)
        keywords: Search keywords for retrieval
        related_functions: Related Fenic functions (e.g., ["semantic.extract", "DataFrame.select"])
        
    Returns:
        str: The ID of the stored learning entry
    """
    # Initialize table if it doesn't exist
    initialize_learnings_table(session)
    
    # Generate unique ID and timestamp
    learning_id = str(uuid.uuid4())
    created_at = datetime.datetime.now().isoformat()
    
    # Convert None to empty lists for proper array handling
    keywords_list = keywords if keywords is not None else []
    related_functions_list = related_functions if related_functions is not None else []
    
    # Create DataFrame with the learning data (using proper arrays)
    learning_data = session.create_dataframe([{
        "id": learning_id,
        "question": question,
        "answer": answer,
        "learning_type": learning_type,
        "keywords": keywords_list,  # Store as actual array
        "related_functions": related_functions_list,  # Store as actual array
        "created_at": created_at
    }])
    
    # Add embeddings for semantic search
    learning_with_embeddings = learning_data.select(
        fc.col("id"),
        fc.col("question"),
        fc.col("answer"),
        fc.col("learning_type"),
        fc.col("keywords"),
        fc.col("related_functions"),
        fc.col("created_at"),
        fc.semantic.embed(fc.col("question")).alias("question_embedding"),
        fc.semantic.embed(fc.col("answer")).alias("answer_embedding"),
        # Create combined embedding for better search
        fc.semantic.embed(
            fc.text.concat(
                fc.col("question"), 
                fc.lit(" "), 
                fc.col("answer"), 
                fc.lit(" "), 
                fc.text.array_join(fc.col("keywords"), " ")
            )
        ).alias("combined_embedding")
    )
    
    # Store in the learnings table
    learning_with_embeddings.write.save_as_table("learnings", mode="append")
    
    return learning_id

In [ ]:
initialize_learnings_table(session)

learning_id1 = store_learning(
        session,
        question="How do I extract structured data from text in Fenic?",
        answer="Use semantic.extract() with either ExtractSchema or Pydantic models. Only basic scalar types are supported.",
        learning_type="solution",
        keywords=["extract", "structured", "semantic", "schema"],  # Proper list
        related_functions=["semantic.extract", "ExtractSchema", "ExtractSchemaField"]  # Proper list
    )


Submitting requests for batch: e4bb732e-6a52-4afc-ab12-57030d064274:   0%|          | 0/1 [00:00<?, ?req/s]
Submitting requests for batch: 1087441d-56f4-483d-9b97-a5479c72bc0f: 100%|██████████| 1/1 [00:00<00:00, 159.16req/s, estimated_input_tokens=12, estimated_output_tokens=0]
Submitting requests for batch: 1941dcc0-4db4-4d67-9b28-895926317b2b: 100%|██████████| 1/1 [00:00<00:00, 283.59req/s, estimated_input_tokens=37, estimated_output_tokens=0]
Submitting requests for batch: e4bb732e-6a52-4afc-ab12-57030d064274: 100%|██████████| 1/1 [00:00<00:00, 338.85req/s, estimated_input_tokens=21, estimated_output_tokens=0]
Awaiting responses for batch 1087441d-56f4-483d-9b97-a5479c72bc0f (model: text-embedding-3-large):   0%|          | 0/1 [00:00<?, ?res/s]
Awaiting responses for batch 1941dcc0-4db4-4d67-9b28-895926317b2b (model: text-embedding-3-large): 100%|██████████| 1/1 [00:00<00:00,  1.68res/s]

Awaiting responses for batch e4bb732e-6a52-4afc-ab12-57030d064274 (model: text-embedding-3-la

In [64]:
def search_combined(query: str, max_results: int = 30, include_learnings: bool = True) -> str:
    """
    Search the Fenic codebase and learned solutions for relevant information.

    This tool automatically searches both:
    1. Learned solutions from past interactions (weighted higher) using semantic similarity
    2. Fenic API documentation and code using text matching

    Learnings appear first when relevant, especially corrections to common mistakes.

    Args:
        query: Search term or regex pattern
        max_results: Maximum total results to return
        include_learnings: Whether to include learned solutions (default: True)

    Returns:
        Search results with learned solutions prioritized, followed by API documentation

    Examples:
        - Simple search: "join"
        - Regex search: "semantic.*extract"
        - Search for specific terms: "DataFrame"
    """
    try:

        # Prepare regex pattern for case-insensitive search
        pattern = f"(?i){query}"

        # Search learnings first if they exist and are requested
        learnings_results = None
        if include_learnings and session.catalog.does_table_exist("learnings"):
            try:
                learnings_df = session.table("learnings")

                # Use semantic search with embeddings!
                # Embed the query and compare with stored embeddings
                learnings_with_similarity = learnings_df.select(
                    fc.col("question"),
                    fc.col("answer"),
                    fc.col("learning_type"),
                    fc.col("keywords"),
                    fc.col("related_functions"),
                    # Compute similarity between query and combined embedding
                    fc.embedding.compute_similarity(
                        fc.col("combined_embedding"),
                        fc.embeddings(fc.lit(query), model_alias="large"),
                        metric="cosine"
                    ).alias("semantic_score")
                )

                # Filter by minimum similarity threshold (e.g., 0.7)
                # and combine with text matching for hybrid search
                learnings_search = learnings_with_similarity.filter(
                    (fc.col("semantic_score") > 0.7) |
                    fc.col("question").rlike(pattern) |
                    fc.col("answer").rlike(pattern) |
                    fc.array_contains(fc.col("keywords"), query)
                )

                # Score learnings with semantic similarity as primary factor
                score_expr = fc.col("semantic_score") * 200  # Semantic similarity weighted highly

                # Add bonuses for text matches
                score_expr = fc.when(fc.col("question").rlike(pattern), score_expr + 50).otherwise(score_expr)
                score_expr = fc.when(fc.col("answer").rlike(pattern), score_expr + 30).otherwise(score_expr)

                # Boost corrections
                score_expr = fc.when(fc.col("learning_type") == "correction", score_expr * 1.5).otherwise(score_expr)

                learnings_scored = learnings_search.select(
                    fc.col("question"),
                    fc.col("answer"),
                    fc.col("learning_type"),
                    fc.col("keywords"),
                    fc.col("related_functions"),
                    fc.col("semantic_score"),
                    score_expr.alias("score")
                )

                # Sort learnings by score and limit
                learnings_sorted = learnings_scored.order_by(fc.col("score").desc()).limit(max_results)
                learnings_results = learnings_sorted.to_pydict()

            except Exception as e:
                # If semantic search fails, fall back to text-only search
                print(f"Warning: Semantic search failed, using text search: {e}")

                # Fallback text-only search
                learnings_search = learnings_df.filter(
                    fc.col("question").rlike(pattern) |
                    fc.col("answer").rlike(pattern) |
                    fc.array_contains(fc.col("keywords"), query)
                )

                learnings_scored = learnings_search.select(
                    fc.col("question"),
                    fc.col("answer"),
                    fc.col("learning_type"),
                    fc.col("keywords"),
                    fc.col("related_functions"),
                    fc.lit(None).alias("semantic_score"),
                    fc.when(fc.col("learning_type") == "correction", fc.lit(200))
                      .when(fc.col("question").rlike(pattern), fc.lit(150))
                      .otherwise(fc.lit(100)).alias("score")
                )

                learnings_sorted = learnings_scored.order_by(fc.col("score").desc()).limit(max_results)
                learnings_results = learnings_sorted.to_pydict()

        # Regular API search (text-based, as before)
        api_df = session.table("api_df")

        # Filter only public API elements
        api_df = api_df.filter(
            (fc.col("is_public") == True) &
            (~fc.col("qualified_name").contains("._"))
        )

        # Search across all text fields
        api_search = api_df.filter(
            fc.col("name").rlike(pattern) |
            fc.col("qualified_name").rlike(pattern) |
            (fc.col("docstring").is_not_null() & fc.col("docstring").rlike(pattern)) |
            (fc.col("annotation").is_not_null() & fc.col("annotation").rlike(pattern)) |
            (fc.col("returns").is_not_null() & fc.col("returns").rlike(pattern))
        )

        # Score API results
        api_scored = api_search.select(
            fc.col("type"),
            fc.col("name"),
            fc.col("qualified_name"),
            fc.col("docstring"),
            fc.when(fc.col("name").rlike(pattern), fc.lit(10))
              .when(fc.col("qualified_name").rlike(pattern), fc.lit(5))
              .otherwise(fc.lit(1)).alias("score")
        )

        # Sort API results by score and limit
        # If we have learnings, reduce API results to make room
        api_limit = max_results
        if learnings_results and len(learnings_results.get('question', [])) > 0:
            api_limit = max(10, max_results - len(learnings_results['question']))

        api_sorted = api_scored.order_by([fc.col("score").desc(), fc.col("type"), fc.col("name")]).limit(api_limit)
        api_results = api_sorted.to_pydict()

        # Format output
        output = f"# Search Results for: `{query}`\n\n"

        # Count total results
        total_results = 0
        if learnings_results:
            total_results += len(learnings_results.get('question', []))
        total_results += len(api_results.get('name', []))

        output += f"Found {total_results} matches\n\n"

        if total_results == 0:
            output += "No results found. Try:\n"
            output += "- Different keywords (e.g., 'extract', 'semantic', 'DataFrame')\n"
            output += "- Regex patterns (e.g., 'join.*semantic')\n"
            return output

        # Show learnings first if any
        if learnings_results and len(learnings_results.get('question', [])) > 0:
            output += "## 📚 Learned Solutions\n\n"

            for i in range(len(learnings_results['question'])):
                # Format learning display
                learning_type = learnings_results['learning_type'][i]
                if learning_type == "correction":
                    output += f"### ⚠ Correction: {learnings_results['question'][i]}\n"
                else:
                    output += f"### 💡 {learnings_results['question'][i]}\n"

                # Show semantic similarity score if available
                if learnings_results.get('semantic_score') and learnings_results['semantic_score'][i]:
                    output += f"*Relevance: {learnings_results['semantic_score'][i]:.2f}*\n\n"

                output += f"{learnings_results['answer'][i]}\n"

                # Add metadata if available
                if learnings_results.get('keywords') and learnings_results['keywords'][i] and len(learnings_results['keywords'][i]) > 0:
                    output += f"\n**Keywords**: {', '.join(learnings_results['keywords'][i])}\n"
                if learnings_results.get('related_functions') and learnings_results['related_functions'][i] and len(learnings_results['related_functions'][i]) > 0:
                    output += f"**Related Functions**: {', '.join(learnings_results['related_functions'][i])}\n"
                output += "\n---\n\n"

        # Show API results
        if len(api_results.get('name', [])) > 0:
            output += "## 📖 API Documentation\n"

            # Group by type
            current_type = None
            for i in range(len(api_results['name'])):
                if api_results['type'][i] != current_type:
                    current_type = api_results['type'][i]
                    output += f"\n### {current_type.capitalize()}s\n"

                # Format each result
                output += f"\n**`{api_results['name'][i]}`** - `{api_results['qualified_name'][i]}`\n"

                # Add docstring if available
                if api_results.get('docstring') and api_results['docstring'][i]:
                    output += f"{api_results['docstring'][i]}\n"

        # Add note about storing learnings if only API results found
        if len(api_results.get('name', [])) > 0 and (not learnings_results or len(learnings_results.get('question', [])) == 0):
            output += "\n---\n"
            output += "💡 **Tip**: If you discover a useful pattern or correction while using these APIs, "
            output += "it will be automatically saved and appear in future searches.\n"

        return output

    except Exception as e:
        import traceback
        return f"Search error: {str(e)}\n{traceback.format_exc()}"

In [65]:
search_combined("how should I use semantic extract?")

'Search error: `lit` failed to infer type for value `None`\nTraceback (most recent call last):\n  File "/var/folders/b_/wgdf93_52y177vcb2ms6n4340000gn/T/ipykernel_909/1715147916.py", line 46, in search_combined\n    fc.embeddings(fc.lit(query), model_alias="large"),\n    ^^^^^^^^^^^^^\nAttributeError: module \'fenic\' has no attribute \'embeddings\'\n\nDuring handling of the above exception, another exception occurred:\n\nTraceback (most recent call last):\n  File "/Users/kostaspardalis/Projects/fenic/src/fenic/api/functions/core.py", line 45, in lit\n    inferred_type = infer_dtype_from_pyobj(value)\n                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/kostaspardalis/Projects/fenic/src/fenic/core/_utils/type_inference.py", line 32, in infer_dtype_from_pyobj\n    raise TypeInferenceError("Null value; please provide a concrete type", path)\nfenic.core._utils.type_inference.TypeInferenceError: Null value; please provide a concrete type\n\nThe above exception was the direct c

In [59]:
session.table("learnings").show()

┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ question  ┆ answer    ┆ learning_ ┆ … ┆ created_a ┆ question_ ┆ answer_em ┆ combined │
│           ┆           ┆           ┆ type      ┆   ┆ t         ┆ embedding ┆ bedding   ┆ _embeddi │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ ng       │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 3098febf- ┆ How do I  ┆ Use seman ┆ solution  ┆ … ┆ 2025-07-0 ┆ [-0.00204 ┆ [-0.00644 ┆ [-0.0087 │
│ dae8-4bbb ┆ extract   ┆ tic.extra ┆           ┆   ┆ 6T10:55:2 ┆ 6, -0.018 ┆ 3,        ┆ 51, -0.0 │
│ -87cf-ffd ┆ structure ┆ ct() with ┆           ┆   ┆ 7.995913  ┆ 858, …    ┆ 0.008596, ┆ 13217, … │
│ 2adb79e5d ┆ d data    ┆ either    ┆           ┆   ┆           ┆ -0.031456 ┆ … -0.0069 ┆ -0.02544 │
│           ┆ from text ┆ ExtractSc ┆           ┆   ┆           ┆ ]         ┆ 46]       ┆ ]